# 04 Prediction Error And Attention-Like Signal

Use absolute prediction error as a simple surprise proxy after a signal change.

In [ ]:
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from predictive_brain_lab.data import generate_frequency_shift_signal
from predictive_brain_lab.datasets import SlidingWindowDataset
from predictive_brain_lab.metrics import absolute_prediction_error, detect_surprise_events
from predictive_brain_lab.models import MLPPredictor
from predictive_brain_lab.train import predict_torch_model, train_torch_model
from predictive_brain_lab.viz import plot_predictions_and_errors

In [ ]:
signal, metadata = generate_frequency_shift_signal(
    n_steps=1200,
    frequency_before=2.0,
    frequency_after=6.0,
    change_point=800,
    seed=7,
)
dataset = SlidingWindowDataset(signal, window_size=50)
pre_change_indices = [i for i, target_idx in enumerate(dataset.target_indices) if target_idx < metadata['change_point']]
split = int(0.8 * len(pre_change_indices))

train_loader = DataLoader(Subset(dataset, pre_change_indices[:split]), batch_size=64, shuffle=True)
val_loader = DataLoader(Subset(dataset, pre_change_indices[split:]), batch_size=64)
full_loader = DataLoader(dataset, batch_size=64)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = MLPPredictor(input_size=50, hidden_size=64)
train_torch_model(model, train_loader, val_loader, epochs=10, lr=1e-3, device=device)
y_true, y_pred = predict_torch_model(model, full_loader, device=device, model_type='mlp')
errors = absolute_prediction_error(y_true, y_pred)
surprise_events = detect_surprise_events(errors, percentile=95)

plot_predictions_and_errors(
    signal=signal,
    target_indices=dataset.target_indices,
    predictions=y_pred,
    errors=errors,
    surprise_events=surprise_events,
    change_point=metadata['change_point'],
);